# Class Test CA1: Individual Assessment
## Assignment Details
**Module Title:** Data Analytics
**CRN:** 23631
**Student:** Rory Patrick Sheridan - A00082654
**Dataset:**
Annual observations for 25 existing branches of a service company over 2020-2026, recording revenue, local income, competition, customer rating, staff quality, branch type, region and monthly footfall, `ServiceLocation.csv`.
**Task:**
Answer the multiple-answer questions, develop a regression model that predicts next-year branch revenue and use it to recommend an investment site, and apply Bayes' theorem to a fraud-detection decision.

Github Repository:
https://github.com/Ri-Dearg/data-test
Github Notebook:
https://github.com/Ri-Dearg/data-test/blob/main/ca1_class_test.ipynb

### Note
- This notebook follows the brief's sections and question numbers. Each answer is in the markdown cell under its question, with the supporting code and output below it.
- Coefficients are reported to 4 decimal places, and the Question 7 predictions are calculated from those reported values.

## Table of Contents

<details>
<summary><a href="#tools--libraries">Tools & Libraries</a></summary>

- [Libraries](#libraries)
- [Utilities](#utilities)
- [Global Variables](#global-variables)

</details>

<details>
<summary><a href="#section-a-multiple-answer-questions">Section A: Multiple-Answer Questions</a></summary>

- [Question 1](#question-1)
- [Question 2](#question-2)
- [Question 3](#question-3)
- [Question 4](#question-4)
- [Question 5](#question-5)

</details>

<details>
<summary><a href="#section-b-applied-data-analysis">Section B: Applied Data Analysis</a></summary>

- [Dataset: ServiceLocation](#dataset-servicelocation)
- [Question 6: Understand and Model the Data](#question-6-understand-and-model-the-data)
- [Question 7: Make a Management Decision](#question-7-make-a-management-decision)

</details>

<details>
<summary><a href="#section-c-bayes-theorem-and-decision-making">Section C: Bayes' Theorem and Decision-Making</a></summary>

- [Question 8](#question-8)

</details>

<details>
<summary><a href="#references">References</a></summary>

- [AI Use Disclaimer](#ai-use-disclaimer)

</details>

## Tools & Libraries
1. [Matplotlib](https://matplotlib.org) - For plotting graphs.
2. [Numpy](https://numpy.org) - Calculations
3. [Pandas](https://pandas.pydata.org) - Dataframes and data manipulation
4. [Scikit-learn](https://scikit-learn.org/stable/) - For mutual information
5. [Scipy](https://scipy.org) - For the Shapiro-Wilk test, Q-Q plot and t-distribution
6. [Seaborn](https://seaborn.pydata.org) - Plot library
7. [Statsmodels](https://www.statsmodels.org/stable/index.html) - For regression models and their diagnostics
8. [Jupyter](https://jupyter.org) - Notebook Tool

### Libraries
Libraries used.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

## Section A: Multiple-Answer Questions

### Question 1
A positive numerical variable has Mean = 82, Median = 54, and Skewness = +1.7. Which statements are defensible?

**Answer: A, B and D**

### Question 2
For two variables, Pearson correlation is r = 0.03, but the scatterplot shows a clear curved relationship. Which statements are correct?

**Answer: B and C**

### Question 3
A regression model gives R² = 0.91, but its residual plot shows a clear repeating pattern over time. Which statements are defensible?

**Answer: B, C and E**

### Question 4
A hypothesis test gives p = 0.018 at α = 0.05. Which statements are defensible?

**Answer: A, C and E**

### Question 5
A categorical variable Region has four categories: Dublin, Cork, Galway and Limerick. Which statements are defensible?

**Answer: A, B and E**

## Section B: Applied Data Analysis
Section B uses `ServiceLocation.csv`. Basic checks of the dataset come first. The treatment of each issue they find is justified at the start of Question 6.

### Dataset: ServiceLocation

#### Data Overview
The dataset has 175 rows: 25 branches, each observed in every year from 2020 to 2026, with 5 branches in each of the 5 regions. Annual revenue ranges from €0.07 million to €2.99 million.

In [2]:
# Region names for the nominal Location_Code, as given in the brief
REGIONS = {1: 'Midlands', 2: 'North', 3: 'East', 4: 'South', 5: 'West'}

# Import the dataset, ordered by branch and year
df = pd.read_csv('ServiceLocation.csv').sort_values(['Branch_ID', 'Year']).reset_index(drop=True)
df

,Branch_ID,Year,Revenue_M_EUR,Local_Income_kEUR,Competition,Customer_Rating,Staff_Quality,Modern_Type,Location_Code,Footfall_Monthly
0,B001,2020,1.798,40.3,8,7.2,7.0,1,2,10890
1,B001,2021,1.484,41.4,8,7.2,6.0,1,2,10289
2,B001,2022,1.583,41.7,7,7.3,6.0,1,2,10888
3,B001,2023,1.729,42.9,7,7.2,6.0,1,2,10782
4,B001,2024,1.740,43.2,7,7.2,6.0,1,2,10605
...,...,...,...,...,...,...,...,...,...,...
170,B025,2022,2.415,45.3,3,7.9,6.0,0,2,11107
171,B025,2023,2.282,45.5,3,7.8,6.0,1,2,11661
172,B025,2024,2.441,46.3,3,8.0,6.0,1,2,11894
173,B025,2025,2.713,46.9,3,7.9,5.0,1,2,12297


In [ ]:
# View data stats
df.describe().round(2)

,Year,Revenue_M_EUR,Local_Income_kEUR,Competition,Customer_Rating,Staff_Quality,Modern_Type,Location_Code,Footfall_Monthly
count,175.00,175.00,174.00,175.00,175.00,174.00,175.00,175.00,175.00
mean,2023.00,1.79,42.24,4.49,6.91,6.60,0.40,3.00,10874.43
std,2.01,0.55,4.31,2.18,0.85,1.21,0.49,1.42,775.82
min,2020.00,0.07,31.50,1.00,0.80,4.00,0.00,1.00,8104.00
25%,2021.00,1.50,39.30,3.00,6.40,6.00,0.00,2.00,10457.50
50%,2023.00,1.80,43.15,5.00,6.90,7.00,0.00,3.00,10883.00
75%,2025.00,2.17,45.30,6.00,7.30,7.00,1.00,4.00,11371.00
max,2026.00,2.99,52.20,10.00,8.60,9.00,1.00,5.00,12828.00


In [5]:
# Branches per region
print(df.groupby('Location_Code')['Branch_ID'].nunique().rename(index=REGIONS))

Location_Code
Midlands    5
North       5
East        5
South       5
West        5
Name: Branch_ID, dtype: int64


#### Types
`Branch_ID` is text and every other variable is numeric. `Location_Code` is stored as a number but is a nominal region code, and `Modern_Type` is a 0/1 indicator (Question 6 b). `Local_Income_kEUR` and `Staff_Quality` each have one missing value.

In [6]:
# Explore structure of the data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 175 entries, 0 to 174
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Branch_ID          175 non-null    str    
 1   Year               175 non-null    int64  
 2   Revenue_M_EUR      175 non-null    float64
 3   Local_Income_kEUR  174 non-null    float64
 4   Competition        175 non-null    int64  
 5   Customer_Rating    175 non-null    float64
 6   Staff_Quality      174 non-null    float64
 7   Modern_Type        175 non-null    int64  
 8   Location_Code      175 non-null    int64  
 9   Footfall_Monthly   175 non-null    int64  
dtypes: float64(4), int64(5), str(1)
memory usage: 13.8 KB
